In [4]:
# imports
import os
import glob
from dotenv import load_dotenv
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAI
from langchain_chroma import Chroma
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_models import ChatOllama
from sklearn.metrics.pairwise import cosine_similarity


In [5]:
load_dotenv(override=True)

False

In [6]:
# if you want to use model with api keys (openai, google, deepseek, anthropic) otherwise just comment them
# if so, you need to set the api keys in the .env file as below | create a text file, rename it .env and put it in the root of the project
# OPENAI_API_KEY=xxxxxxxxx
# ANTHROPIC_API_KEY=xxxxxxxxx
# DEEPSEEK_API_KEY=xxxxxxxxx
# GOOGLE_API_KEY=xxxxxxxxx
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

OpenAI API Key not set
Anthropic API Key not set (and this is optional)
Google API Key not set (and this is optional)
DeepSeek API Key not set (and this is optional)


In [7]:
db_name = "vector_store"

# I used the following prompt in chatgpt to generate text related to A FAKE company called CAPCO!

generate 3 text files, each contains 600 words for a company called CAPCO Intelligent Services. I will use these generated data in some RAG and LLMs examples to my students to simulate an existing company. the company located in Nablus - Palestine, established in 2018 to offer AI services. the first file generate text related to ABOUT the company including and not limited to history, mission, vision and objectives. the second file, generate text related to the 5 departments and 15 staff members in the company, provide middle east names for the staff, their positions and roles and a bio in two lines for each person, the CEO name of the company is Eng. Mohammad Rasheed. the third file generate text related to the 6 services of the company such that it offers ERP solutions, AI solutions, Business intelligence solutions, cyber security and so on. In each file generate texts in paragraphs seperated by new lines. generate the text in professional way to simulate an existing running company. export the files as name.txt. It is important to generate coherent text and do not repeat text in each file

In [8]:
# Read in documents using LangChain's loaders
# Take everything in all the sub-folders of our knowledgebase
folders = glob.glob("knowledge-base3/")
text_loader_kwargs = {'encoding': 'utf-8'}
# text_loader_kwargs={'autodetect_encoding': True}
documents = []
for folder in folders:  
    loader = DirectoryLoader(folder, glob="**/*.txt", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    for doc in folder_docs:
        file_name = os.path.basename(doc.metadata["source"])
        doc.metadata["doc_type"] = os.path.splitext(file_name)[0]
        documents.append(doc)

In [9]:
len(documents)

1

In [10]:
documents

[Document(metadata={'source': 'knowledge-base3\\knowledge-base3\\classes_it.txt', 'doc_type': 'classes_it'}, page_content='القاعة: 140120. المبنى: كلية العلوم وتكنولوجيا المعلومات والبصريات. الطابق: غير متوفر. وصف المكان: ملغي. الحرم: الحرم الجامعي الجديد - نابلس\nالقاعة: 140210. المبنى: كلية العلوم وتكنولوجيا المعلومات والبصريات. الطابق: غير متوفر. وصف المكان: ملغي. الحرم: الحرم الجامعي الجديد - نابلس\nالقاعة: 141010. المبنى: كلية العلوم وتكنولوجيا المعلومات والبصريات. الطابق: غير متوفر. وصف المكان: مكتبة. الحرم: الحرم الجامعي الجديد - نابلس\nالقاعة: 141020. المبنى: كلية العلوم وتكنولوجيا المعلومات والبصريات. الطابق: غير متوفر. وصف المكان: مكتب. الحرم: الحرم الجامعي الجديد - نابلس\nالقاعة: 141050. المبنى: كلية العلوم وتكنولوجيا المعلومات والبصريات. الطابق: الطابق الأول. وصف المكان: قاعة أنشطة. الحرم: الحرم الجامعي الجديد - نابلس\nالقاعة: 141090. المبنى: كلية العلوم وتكنولوجيا المعلومات والبصريات. الطابق: غير متوفر. وصف المكان: مكتب مدرس. الحرم: الحرم الجامعي الجديد - نابلس\nالقاعة: 14

In [12]:
documents[0].metadata

{'source': 'knowledge-base3\\knowledge-base3\\classes_it.txt',
 'doc_type': 'classes_it'}

In [13]:
#chunk size = 5000 character || chunk_overlap => take the next chunk from 5000-300
text_splitter = CharacterTextSplitter(separator="\n", chunk_size=5000, chunk_overlap=300) 
chunks = text_splitter.split_documents(documents)

In [14]:
len(chunks)

6

In [15]:
chunks[5]

Document(metadata={'source': 'knowledge-base3\\knowledge-base3\\classes_it.txt', 'doc_type': 'classes_it'}, page_content='القاعة: 14G5130. المبنى: كلية العلوم وتكنولوجيا المعلومات والبصريات. الطابق: غير متوفر. وصف المكان: لايوجد. الحرم: الحرم الجامعي الجديد - نابلس\nالقاعة: 14G6080. المبنى: كلية العلوم وتكنولوجيا المعلومات والبصريات. الطابق: غير متوفر. وصف المكان: ملغي. الحرم: الحرم الجامعي الجديد - نابلس\nالقاعة: 14b4262. المبنى: كلية العلوم وتكنولوجيا المعلومات والبصريات. الطابق: غير متوفر. وصف المكان: لايوجد. الحرم: الحرم الجامعي الجديد - نابلس')

In [16]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

Document types found: classes_it


In [17]:
# embeddings = OpenAIEmbeddings()
# If you would rather use the free Vector Embeddings from HuggingFace sentence-transformers
# Then replace embeddings = OpenAIEmbeddings()
# with:
# A text embedding is a numerical representation of a text, usually as a dense vector of real numbers. 
# It captures semantic meaning in a format models can understand and compare.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\ai_wo\AppData\Local\Temp\ipykernel_18376\516977989.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
# Check if a Chroma Datastore already exists - if so, delete the collection to start from scratch

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

In [19]:
# Create our Chroma vectorstore!
# Put the chunks of data into a Vector Store that associates a Vector Embedding with each chunk
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 6 documents


In [20]:
# To read data from the vector store
collection = vectorstore._collection
ds = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(ds['embeddings'])
docs = ds['documents']

doc_types = [metadata['doc_type'] for metadata in ds['metadatas']]
# colors = [['blue', 'green', 'red'][['departments_and_staff', 'about', 'services'].index(t)] for t in doc_types]

In [21]:
# Get one vector and find how many dimensions it has
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"The vectors have {dimensions:,} dimensions")

The vectors have 384 dimensions


t-SNE stands for t-distributed Stochastic Neighbor Embedding. It is a dimensionality reduction algorithm commonly used for visualizing high-dimensional data.

t-SNE takes data with many dimensions (e.g., 100D word embeddings or image features) and reduces it to 2 or 3 dimensions, making it easier to visualize patterns and clusters.
Example: Visualizing 300-dimensional word embeddings (like Word2Vec or GloVe) in 2D space to understand how similar words are grouped.


In [22]:
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    # marker=dict(size=10, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, docs)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [24]:
# create a new Chat with OpenAI / Ollama

llama_model = ChatOllama(model="llama3.2:latest")

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG
retriever = vectorstore.as_retriever()

# putting it together: set up the conversation chain with the GPT 4o-mini LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llama_model, retriever=retriever, memory=memory)

C:\Users\ai_wo\AppData\Local\Temp\ipykernel_18376\127700709.py:6: LangChainDeprecationWarning:

Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/



In [25]:
query = "what is capco?"

result = conversation_chain.invoke({"question":query})
print(result["answer"])

Capco can refer to different things, but I'll provide a few possible explanations:

1. **Capcom**: Capcom Co., Ltd. is a Japanese multinational video game developer and publisher based in Tokyo, Japan. The company was founded in 1979 and is best known for creating popular franchises such as Resident Evil, Street Fighter, and Devil May Cry.
2. **Cappercolini (Capco)**: Cappercolini (also known as Capco) is an Italian fashion brand that designs and produces high-end women's clothing, shoes, and accessories. The company was founded in 2000 by Giovanni Versace and is part of the Versace Group.
3. **CAPCO**: CAPCO stands for Certified Air Purification Consultant, which is a professional certification awarded to individuals who have demonstrated expertise in air quality assessment, indoor air pollution, and ventilation systems.

If you could provide more context or information about Capco, I'd be happy to try and give a more specific answer!
